# F1 · From a tree to Extra Trees: how much each ingredient gains

**Type**: Compare · **Level**: ●●○ independent · **Time**: about 50 minutes · **core path ★**

## Objective

Compare four models **built one on top of the other** on three datasets (wines, tumors, digits): a single tree, bagging of trees (variance reduced by the average), the random forest (bagging + random features) and Extra Trees (random thresholds on top of random features). Measure accuracy in repeated cross-validation and time,
and quantify the gain of each ingredient.

## Dataset

`load_wine`, `load_breast_cancer`, `load_digits`; validation `RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=0)`; 100 trees, `random_state=0`.

## Educational purpose

See the story of the module in numbers: each ingredient (averaging, decorrelation, random thresholds) adds a gain. And remember that small differences (tumors: 0.5 points between forest and Extra Trees) are noise, while the jump from the single tree is large.

## Task

1. For each dataset and model (`tree` = `DecisionTreeClassifier`, `bagging` = `BaggingClassifier(DecisionTreeClassifier(), 100)`, `forest` = `RandomForestClassifier(100)`, `extra` = `ExtraTreesClassifier(100)`) compute with `cross_validate` `acc` (mean) and `t` (mean total fit time per fold, seconds). All with `random_state=0`, `n_jobs=-1` where it exists. Save `table` (index `(dataset, model)`).
2. Save `gains` (`pd.DataFrame`, index: the datasets; columns `averaging` = bagging − tree, `random_features` = forest − bagging, `random_thresholds` = extra − forest).
3. Save `order_holds` (True if, for **all** the datasets, `tree < bagging <= forest + 0.005` and `extra >= forest − 0.005`) and `dominant_gain` (name of the column of `gains` with the largest sum).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer, load_digits, load_wine
from sklearn.ensemble import BaggingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier

datasets = {"wine": load_wine(return_X_y=True), "tumors": load_breast_cancer(return_X_y=True), "digits": load_digits(return_X_y=True)}
models = {
    "tree": lambda: DecisionTreeClassifier(random_state=0),
    "bagging": lambda: BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=0, n_jobs=-1),
    "forest": lambda: RandomForestClassifier(100, random_state=0, n_jobs=-1),
    "extra": lambda: ExtraTreesClassifier(100, random_state=0, n_jobs=-1),
}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=0)

## Your work

In [ ]:
rows = {}
for name, (X, y) in datasets.items():
    for model_name, create in models.items():
        ...
table = pd.DataFrame(rows).T
table.round(3)

In [ ]:
gains = None
order_holds = None
dominant_gain = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`r = cross_validate(create(), X, y, cv=cv)`; `acc = r["test_score"].mean()`; `t = r["fit_time"].mean()`. Gains: `table["acc"].unstack().loc[list(datasets)]` has the datasets as rows (in the original order) and the models as columns: `u["bagging"] - u["tree"]`, etc.'):
    check(table.shape == (12, 2) and list(table.columns) == ["acc", "t"], "12 rows (3 datasets x 4 models) and columns acc, t", "one row per (dataset, model) pair")
    check(list(gains.columns) == ["averaging", "random_features", "random_thresholds"] and list(gains.index) == list(datasets), "gains per dataset and per ingredient", "columns: averaging, random_features, random_thresholds")
    check((gains["averaging"] > 0.02).all() and dominant_gain == "averaging", "averaging the trees always gains over 2 points and it is the dominant gain", "bagging - tree")
    check(gains["random_features"].min() > -0.002 and gains["random_features"].max() < 0.03 and (gains["random_features"] >= 0).sum() >= 2, "random features gain little (0.5-2.5 points) and do not make things worse", "forest - bagging")
    check(order_holds is True and table.loc[("digits", "extra"), "acc"] > table.loc[("digits", "tree"), "acc"] + 0.1, "the order tree < bagging <= forest <= extra holds on all the datasets; on the digits the single tree is 10 points below Extra Trees", "compare the rows")
    check(table.loc[("wine", "tree"), "t"] < table.loc[("wine", "bagging"), "t"] / 3, "the single tree is much faster than the bagging (over three times)", "the cost grows with the number of trees")
done()

---
The commented solution is in `solutions/F_compare/F1_the_ladder_from_tree_to_extra_trees.ipynb`. Open it after you pass the check, or after you have really tried.